In [3]:
import pandas as pd
import os
import re
import xmltodict
from typing import List

import vertexai 
from google.cloud import bigquery, storage
from google import genai

from search_eval_utils import access_secret_version


In [4]:
project_looker = 'Looker-Studio-Pro'
bucket = 'dodge_full_dump' 
filename = 'Delta/Unified_20250218.xml'

PROJECT_ID = 'proj-sales-recommender-dev'
LOCATION = 'us-central1'
DATASET = 'sales_recommender_dev'
TABLE = 'relevant_materials'

# Gemini vars 
SECRET_NAME = "gemini_api_key"
MODEL_ID = "gemini-2.0-flash-001"

In [5]:
def download_from_gcs(storage_client, bucket_name: str, filename: str, local_dir: str ='data/') -> str:
    try:
        # Initialize GCS client
        bucket = storage_client.bucket(bucket_name)
        blob = bucket.blob(filename)

        os.makedirs(local_dir, exist_ok=True)  # Ensure the current directory exists
        
        # Download XML content to a local file
        base_filename = os.path.basename(filename)
        local_xml_file_path = os.path.join(local_dir, base_filename)

        # Download XML content to local file
        blob.download_to_filename(local_xml_file_path)

        return local_xml_file_path

    except Exception as e:
        print(f"An error occurred: {e}")
        return None

In [6]:
client = storage.Client(project=project_looker)
local_xml_file_path = download_from_gcs(storage_client=client, bucket_name=bucket, filename=filename)

In [7]:
local_xml_file_path = 'data/Unified_20250218.xml'
with open(local_xml_file_path, 'r', encoding='utf-8') as xml_file:
    data_dict = xmltodict.parse(xml_file.read())

In [8]:
print("Number of projects: ", len(data_dict['Projects']['Project']))

Number of projects:  1039


In [9]:
# Make DataFrame of projects
df = pd.DataFrame(data_dict['Projects']['Project'])
df.head()

,DRNumber,VersionNumber,DodgeReportType,ProjectURL,EmailAddress,DateOfFirstExport,DateOfLastExport,DateOfCurrentExport,FirstIssueDate,LastIssueDate,...,DeliverySystem,FrameType,StoriesAbove,StoriesBelow,NoOfBuildings,StructuralInfo,FeaturesInfo,SpecAlerts,SearchNames,Companies
0,202500063662,1,Project,https://apps.construction.com/projects/2025000...,intman.unifieddoor@dodgepipeline.com,02/18/2025,None,02/18/2025,2/17/2025,None,...,None,None,None,None,None,None,"Two story, 20,000 square ft office building",None,07292024_Historical_Files,"{'Company': [{'FactorType': 'Civil Engineer', ..."
1,202500063661,1,Project,https://apps.construction.com/projects/2025000...,intman.unifieddoor@dodgepipeline.com,02/18/2025,None,02/18/2025,2/17/2025,None,...,None,None,None,None,None,None,"4,332-square-foot, one-story convenience store...",None,07292024_Historical_Files,"{'Company': {'FactorType': 'Owner', 'FactorKey..."
2,202500063660,1,Project,https://apps.construction.com/projects/2025000...,intman.unifieddoor@dodgepipeline.com,02/18/2025,None,02/18/2025,2/17/2025,None,...,None,None,3,None,5,None,"Four, three story apartment buildings 178 unit...",None,07292024_Historical_Files,"{'Company': [{'FactorType': 'Architect', 'Fact..."
3,202500063659,1,Project,https://apps.construction.com/projects/2025000...,intman.unifieddoor@dodgepipeline.com,02/18/2025,None,02/18/2025,2/17/2025,None,...,None,None,None,None,None,None,"restore the existing exterior cladding, fascia...",None,07292024_Historical_Files,"{'Company': {'FactorType': 'Owner', 'FactorKey..."
4,202500063650,1,Project,https://apps.construction.com/projects/2025000...,intman.unifieddoor@dodgepipeline.com,02/18/2025,None,02/18/2025,2/17/2025,None,...,None,None,None,None,None,None,Convert existing house to worker housing,None,07292024_Historical_Files,"{'Company': {'FactorType': 'Owner', 'FactorKey..."


In [10]:
print(df[['FeaturesInfo',]][:10].to_markdown())

|    | FeaturesInfo                                                            |
|---:|:------------------------------------------------------------------------|
|  0 | Two story, 20,000 square ft office building                             |
|  1 | 4,332-square-foot, one-story convenience store and gas station - four   |
|    | fuel pumps - canopy                                                     |
|  2 | Four, three story apartment buildings 178 units - clubhouse - 183       |
|    | street parking spaces - 44 covered parking spaces                       |
|  3 | restore the existing exterior cladding, fascia, soffits, railings,      |
|    | columns and decking by scraping, repairing and repainting in the        |
|    | original color scheme. A new gutter and downspout will be installed on  |
|    | the front porch. Facade windows and doors will be repaired or replaced  |
|    | as needed and new signage will be added.                                |
|  4 | Convert existing hous

In [11]:
# Save FeaturesInfo to a separate DataFrame
features_info_df = df[['FeaturesInfo']].copy()
features_info_df['length'] = features_info_df.apply(lambda x: len(x['FeaturesInfo']) if x['FeaturesInfo'] else 0, axis=1)

features_info_df = features_info_df[features_info_df['length'] > 0].reset_index(drop=True) # Drop empty entries
features_info_df.sort_values(by='length', ascending=False, inplace=True)    # Sort by length

In [12]:
print(features_info_df.head(5)['FeaturesInfo'].to_markdown()) # Longest FeaturesInfo

|     | FeaturesInfo                                                             |
|----:|:-------------------------------------------------------------------------|
| 603 | Installation and inspection of modular offices. SPECIFICATIONS THE       |
|     | ATTACHED SPEC/IFB NO. 136563 IS HEREBY MADE A PART OF THIS BID.          |
|     | Project: 40000-CP GMA SPOT CONSTRUCTION Item No. Qty / UOM               |
|     | Description 1 1 ALL FOR NAICS 238910 Mobilization for construction,      |
|     | for a lump sum price of: 2 1 ALL FOR NAICS 238990 Complete               |
|     | Installation of two (2) 72-foot x 60-foot modular office building (Six   |
|     | 12-foot wide x 60-foot long sections) for the Los Angeles Reservoir      |
|     | Facility located at 13101 Sepulveda Blvd., Los Angeles, CA 91344 in      |
|     | accordance with the requirements set forth in these specifications,      |
|     | for a lump sum price of: 3 1 ALL FOR NAICS 238990 Complete               |
|   

In [13]:
print(features_info_df.tail(5)['FeaturesInfo'].to_markdown()) # Shortest non-empty FeaturesInfo

|     | FeaturesInfo   |
|----:|:---------------|
|  77 | CAFE/OFFICE    |
|  24 | GARAGE COMM    |
| 325 | New Store      |
| 412 | Remodel        |
| 514 | Remodel        |


In [ ]:
# Fetch materials from BigQuery
bigquery_client = bigquery.Client(project=PROJECT_ID)
materials_query = f"SELECT material FROM `{PROJECT_ID}.{DATASET}.{TABLE}`"
rows = bigquery_client.query(materials_query).result()

In [15]:
materials_lst = [row['material'] for row in rows]

In [16]:
def sanitize_enum_name(name: str) -> str: 
    name = re.sub(r'[^a-zA-Z0-9_]', '_', name)  # Replace non-alphanumeric characters with underscores
    if not name or not name[0].isalpha(): 
        name = "MATERIAL_" + name
    return name.upper()

In [ ]:
from enum import Enum 
from pydantic import BaseModel
from typing import Type, TypeVar 

# Create an Enum for all materials
material_enums = {sanitize_enum_name(material): material for material in materials_lst}
MaterialsEnum = Enum('MaterialsEnum', material_enums)

In [ ]:
class MaterialsRelevanceResult(BaseModel): 
    material: MaterialsEnum 
    Reasoning: str

In [19]:
vertexai.init(project=PROJECT_ID, location=LOCATION)

# Init Gemini client 
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/Users/Kaylee.Rosendahl/fbmSalesRecommender/sa-key.json"
os.environ["GEMINI_API_KEY"] = access_secret_version(PROJECT_ID, SECRET_NAME)
gemini_client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

In [20]:
T = TypeVar('T')

def generate_response(gemini_client, model_id: str, dodge_project_json: str,
                        materials: List[str], response_type=Type[T]) -> str:

    prompt = f"""

        You are an expert in construction projects and materials. Your task is to analyze construction project data and identify the necessary materials for the job. 

        **Instructions:**

        1. **Examine the `Project Data`** provided in JSON format. Identify key features and components, such as project title, feature, and structural information. 
        2. **Review the `Available Materials List`**. This list contains the materials you are allowed to select from. 
        3. **Identify Relevant Materials:** Determine which materials from the provided list are essential for the project construction.
        4. **Provide Justification:** For each selected material, provide a concise `reasoning` for its inclusion based on the project data.
        5. **Adhere to Materials List:** Only select materials that are explicitly present in the `Available Materials List`. Do not infer or suggest materials not on this list.

        Return as a JSON object with each relevant material and your reasoning. 

        Example Output:
        [
            {{
                "material": "Siding",
                "reasoning": "Siding is required as it is a new building which needs exterior cladding."
            }},
            {{
                "material": "Door Hardware",
                "reasoning": "Door hardware is necessary since it is a new building."
            }}
            ]

        Input Project Data:
        {dodge_project_json}

        Relevant Materials:
        {materials} 
        
    """ 

    generation_config = { 
        "temperature": 0.7, 
        "candidate_count": 1,
        "max_output_tokens": 2048,
        "response_mime_type": "application/json", 
        "response_schema": response_type
    }

    # Generate content using the model
    response = gemini_client.models.generate_content(
        model=model_id,
        contents=prompt, 
        config=generation_config,
    )

    if issubclass(response_type, Enum): 
        return response_type(response.text.strip())
    else: 
        return response.parsed
    
# Return list of response_type objects
def generate_response_list(gemini_client, model_id: str, dodge_project_json: str, 
                            materials: List[str], response_type=Type[T]): 
    
    return generate_response(
        gemini_client, model_id, dodge_project_json, 
        materials, response_type=list[response_type]
    )

In [21]:
# Fetch sample of projects from Ddoge
sample_projects = df[:5] 
print(sample_projects[['ProjectTitle', 'FeaturesInfo']].to_markdown())

|    | ProjectTitle                                       | FeaturesInfo                                                            |
|---:|:---------------------------------------------------|:------------------------------------------------------------------------|
|  0 | Maxwell Street Office Building                     | Two story, 20,000 square ft office building                             |
|  1 | Byrne Dairy & Deli Convenience Store & Gas Station | 4,332-square-foot, one-story convenience store and gas station - four   |
|    |                                                    | fuel pumps - canopy                                                     |
|  2 | 390 Woodcliff Apartments                           | Four, three story apartment buildings 178 units - clubhouse - 183       |
|    |                                                    | street parking spaces - 44 covered parking spaces                       |
|  3 | Brick Oven Inn Restaurant (facade improvements)    | re

In [22]:
from tqdm import tqdm 
sample_projects['relevant_materials'] = None  # Initialize column for relevant materials

for index, row in tqdm(sample_projects.iterrows(), total=len(sample_projects)):

    dodge_project_json = row 

    try: 
        # Fetch relevant materials 
        response = generate_response_list(
            gemini_client=gemini_client,
            model_id=MODEL_ID,
            dodge_project_json=dodge_project_json,
            materials=materials_lst,
            response_type=MaterialsRelevanceResult
        )

        # Process and save response to df 
        relevant_materials = {} 
        for item in response: 
            relevant_materials[item.material.value] = item.Reasoning

        sample_projects.at[index, 'relevant_materials'] = relevant_materials

    except Exception as e: 
        print(f"Error processing project {index}: {e}")
        row['relevant_materials'] = None

/var/folders/t9/ynlywvys31d_71jprr25h4c00000gn/T/ipykernel_12071/1514423714.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sample_projects['relevant_materials'] = None  # Initialize column for relevant materials


100%|██████████| 5/5 [00:08<00:00,  1.75s/it]


In [23]:
print(sample_projects[['ProjectTitle', 'FeaturesInfo', 'relevant_materials']].to_markdown())

|    | ProjectTitle                                       | FeaturesInfo                                                            | relevant_materials                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   |
|---:|:---------------------------------------------------|:---------------------------------------------------------------